In [15]:
import pandas as pd
import os
import kagglehub
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix



In [12]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = os.path.join(path, "creditcard.csv")

df = pd.read_csv(csv_path)
df = df.sort_values("Time").reset_index(drop=True)

split_index = int(len(df) * 0.75)

df_train = df.iloc[:split_index].copy()
df_stream = df.iloc[split_index:].copy()

print("Total:", len(df))
print("Train:", len(df_train))
print("Stream:", len(df_stream))

print("\nTrain time range:", df_train["Time"].min(), "->", df_train["Time"].max())
print("Stream time range:", df_stream["Time"].min(), "->", df_stream["Time"].max())


Total: 284807
Train: 213605
Stream: 71202

Train time range: 0.0 -> 139320.0
Stream time range: 139321.0 -> 172792.0


In [27]:
X_train = df_train[df_train["Class"] == 0].drop(columns=["Class"])

model = IsolationForest(
    n_estimators=100,
    contamination=0.04,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train)

print("Model trained.")


Model trained.


In [28]:
# Features + Labels für Stream-Daten
X_stream = df_stream.drop(columns=["Class"])
y_stream = df_stream["Class"]

# Prediction (-1 = anomaly, 1 = normal)
pred = model.predict(X_stream)

# Umwandeln in 0/1 (wie Labels)
pred = [1 if p == -1 else 0 for p in pred]

# Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_stream, pred))

print("\nClassification Report:")
print(classification_report(y_stream, pred))



Confusion Matrix:
[[67803  3305]
 [   18    76]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98     71108
           1       0.02      0.81      0.04        94

    accuracy                           0.95     71202
   macro avg       0.51      0.88      0.51     71202
weighted avg       1.00      0.95      0.97     71202

